In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, precision_recall_curve, average_precision_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler, ADASYN, SMOTE
from joblib import Parallel, delayed
from tqdm import tqdm
import random
import warnings

import os

# Limit each parallel process to one thread per library
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch
device = 'cpu'

# Additional control for PyTorch if needed
import torch
torch.set_num_threads(1)

# For reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Load the dataset
data = pd.read_excel(r'C:\Users\Inspiron\OneDrive - Loughborough University\Desktop\PhD\articles\prospective study\results\dataset\class 123\class123_dataset.xlsx')

# Define feature groups (as provided)
feature_groups = {
    'Genotype': [
        'rs11225395', 'rs1144393', 'rs650108', 'rs591058', 'rs2252070', 'rs4986938', 'rs1800012', 'rs4789932', 'rs9340799', 'rs970547', 
        'rs1800795', 'rs13946', 'rs12722', 'class1_SNP_risk_score', 'rs7528684', 'rs4919510', 'rs1937810', 'rs6481512', 'rs1249269', 
        'rs12574452', 'rs12429486', 'rs4454832', 'rs2761884', 'rs62051384', 'rs4362400', 'rs2586488', 'rs2277698', 'rs1045485', 
        'rs143383', 'rs17576', 'rs2305948', 'rs1011814', 'rs11154027', 'rs2234693', 'rs1643821', 'rs2010963', 'rs10263021', 'rs149047058', 
        'rs420257', 'rs42517', 'rs42522', 'rs42531', 'rs413826', 'rs2104772', 'rs1330363', 'class12_SNP_risk_score', 'rs3753841', 
        'rs57104447', 'rs1887632', 'rs4654760', 'rs1137101', 'rs2306033', 'rs2277268', 'rs4988321', 'rs11232681', 'rs1718119', 'rs3751143', 
        'rs1544410', 'rs2228570', 'rs4328262', 'rs1021188', 'rs74544784', 'rs78391032', 'rs77569527', 'rs117544024', 'rs912336', 
        'rs3218791', 'rs911263', 'rs2525504', 'rs17756404', 'rs4903399', 'rs10132091', 'rs17583842', 'rs1676303', 'rs11629171', 
        'rs2281518', 'rs2285053', 'rs71404070', 'rs710079', 'rs2858056', 'rs820218', 'rs3018362', 'rs1800470', 'rs1800469', 'rs25487', 
        'rs25489', 'rs2289360', 'rs183364169', 'rs11177', 'rs6617', 'rs3219008', 'rs13107325', 'rs60713544', 'rs145648292', 'rs4244032', 
        'rs12656106', 'rs3045', 'rs187483', 'rs4701616', 'rs144414988', 'rs1800629', 'rs10484958', 'rs4730153', 'rs1800797', 'rs1554606', 
        'rs2237352', 'rs4725069', 'rs12154667', 'rs1548456', 'rs3216902', 'rs35360670', 'rs13317', 'rs1800972', 'rs7035322', 'rs7021589', 
        'rs72758637', 'rs10759753', 'rs3789870', 'rs1138545', 'rs3196378', 'rs1134170', 'rs10992075', 'rs1590', 'rs144371252', 
        'rs761804508', 'class123_SNP_risk_score', 'sex'
    ],
    'History': [
        'Age', 'lower_limb_days_total', 'average_run_hours', 'average_interval_training_frequency', 'EDEQ_total', 'tracking_period_injury',
        'past_stress_injury', 'LEAF-Q', 'Athlete_Score', 'average_run_frequency', 'past_month_injury'
    ],
    'Phenotype': [
        'hip_abduction_peak_torque', 'total_ad_ab_ratio', 'knee_extension_peak_torque', 'knee_flexion_peak_torque', 'navicular_drop', 
        'navicular_drop_asymmetry', 'Q_angle', 'Q_angle_asymmetry', 'VALR_12', 'Impact_peak_12', 'Duty_factor_12', 'BMI', 'BMD_spine',
        'hip_abduction_peak_torque_asymmetry', 'hip_adduction_peak_torque', 'hip_adduction_peak_torque_asymmetry', 
        'knee_extension_peak_torque_asymmetry', 'knee_flexion_peak_torque_asymmetry', 'total_fl_ex_ratio', 'leg_lean_mass', 
        'hip_abduction_peak_angle', 'hip_abduction_peak_angle_asymmetry', 'hip_adduction_peak_angle', 'hip_adduction_peak_angle_asymmetry', 
        'ad_ab_ratio_asymmetry', 'knee_extension_peak_angle', 'knee_extension_peak_angle_asymmetry', 'knee_flexion_peak_angle', 
        'knee_flexion_peak_angle_asymmetry', 'fl_ex_ratio_asymmetry', 'VILR_10', 'VALR_10', 'VILR_asymmetry_10', 'VALR_asymmetry_10', 
        'Impact_peak_10', 'Impact_peak_asymmetry_10', 'Flight_time_10', 'Contact_time_10', 'Duty_factor_10', 'Step_frequency_10', 
        'Cadence_asymmetry_10', 'Duty_factor_asymmetry_10', 'VILR_12', 'VILR_asymmetry_12', 'VALR_asymmetry_12', 
        'Impact_peak_asymmetry_12', 'Flight_time_12', 'Contact_time_12', 'Step_frequency_12', 'Cadence_asymmetry_12', 
        'Duty_factor_asymmetry_12', 'Alt_strike', 'height', 'Mass', 'thigh_lean_mass', 'thigh_ffmi', 'lower_leg_lean_mass', 
        'lower_leg_ffmi', 'leg_ffmi', 'total_lean_mass', 'total_ffmi', 'calf_size', 'BMD_hip', 'BMD_body',
    ],
    'Behaviour': [
        'fat_intake_avg', 'past_month_distance', 'past_month_ratio', 'SC_past_season', 'non_running_past_season', 'fat_intake_BW', 
        'fat_percentage_avg', 'average_energy_availability', 'protein_intake_BW', 'omega3_intake_BW', 'vitaminD_intake_BW', 
        'vitaminC_intake_BW', 'vitaminE_intake_BW', 'calcium_intake_BW', 'copper_intake_BW', 'iron_intake_BW', 'glycine_intake_BW', 
        'arginine_intake_BW', 'past_month_min', 'past_week_ratio', 'past_month_volume_low', 'past_week_ratio_low', 'past_month_ratio_low', 
        'past_month_volume_moderate', 'past_week_ratio_moderate', 'past_month_ratio_moderate', 'past_month_volume_high', 
        'past_week_ratio_high', 'past_month_ratio_high', 'past_month_volume_very_high', 'past_week_ratio_very_high', 
        'past_month_ratio_very_high', 'past_month_calculated_volume', 'past_week_ratio_calculated_volume', 
        'past_month_ratio_calculated_volume', 'resistance_training_past_month', 'resistance_training_past_season', 
        'bodyweight_exercises_past_month', 'bodyweight_exercises_past_season', 'core_stability_past_month', 'core_stability_past_season', 
        'balance_training_past_month', 'balance_training_past_season', 'plyometrics_past_month', 'plyometrics_past_season', 
        'drills_past_month', 'drills_past_season', 'circuit_training_past_month', 'circuit_training_past_season', 'barefoot_past_month', 
        'barefoot_past_season', 'stretching_past_month', 'stretching_past_season', 'SC_past_month', 'non_running_past_month'
    ]
}

# Define predictors and outcome
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome

# Ensure that the feature groups exist in the dataset
for group in feature_groups:
    feature_groups[group] = [feature for feature in feature_groups[group] if feature in X.columns]

num_features = len(X)
feature_to_idx = {feature: idx for idx, feature in enumerate(X)}

# Initialize adjacency matrix with zeros
adjacency = np.zeros((num_features, num_features), dtype=np.float32)

# Define groups
genotype_features = feature_groups['Genotype']
history_features = feature_groups['History']
phenotype_features = feature_groups['Phenotype']
behaviour_features = feature_groups['Behaviour']

# Get indices for each group
genotype_indices = [feature_to_idx[feat] for feat in genotype_features]
history_indices = [feature_to_idx[feat] for feat in history_features]
phenotype_indices = [feature_to_idx[feat] for feat in phenotype_features]
behaviour_indices = [feature_to_idx[feat] for feat in behaviour_features]

# Genotype nodes: connect to every node except themselves
for i in genotype_indices:
    for j in range(num_features):
        if j != i:
            adjacency[i, j] = 1.0

# History nodes: connect to every node except genotype nodes and themselves
for i in history_indices:
    for j in range(num_features):
        if j not in genotype_indices and j != i:
            adjacency[i, j] = 1.0

# Phenotype nodes: connect to every node except genotype, history nodes, and themselves
for i in phenotype_indices:
    for j in range(num_features):
        if j not in genotype_indices and j not in history_indices and j != i:
            adjacency[i, j] = 1.0

# Behaviour nodes: connect to all other behaviour nodes except themselves
for i in behaviour_indices:
    for j in behaviour_indices:
        if j != i:
            adjacency[i, j] = 1.0

# Convert adjacency to tensor
adjacency_mask = torch.tensor(adjacency, dtype=torch.float32)

class CustomDataset(Dataset):
    def __init__(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.X = X.values.astype(np.float32)
        elif isinstance(X, np.ndarray):
            self.X = X.astype(np.float32)
        else:
            raise TypeError("X should be a pandas DataFrame or a NumPy array.")
        self.y = y.astype(np.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class CustomNetwork(nn.Module):
    def __init__(self, num_features, adjacency_mask):
        super(CustomNetwork, self).__init__()
        self.num_features = num_features

        # Register the fixed adjacency mask as a buffer (non-trainable)
        self.register_buffer('mask', adjacency_mask.float())  # Shape: (num_features, num_features)

        # Initialize connection weights (W_ij: from node j to node i)
        self.W = nn.Parameter(torch.randn(num_features, num_features) * 0.01)

        # Initialize bias for each node
        self.bias = nn.Parameter(torch.zeros(num_features))

        # Initialize output weights (from each node to the final output)
        self.output_weights = nn.Parameter(torch.randn(num_features) * 0.01)

        # Add Batch Normalization
        self.batch_norm = nn.BatchNorm1d(num_features)

    def forward(self, x, threshold=None):
        """
        x: input tensor of shape (batch_size, num_features)
        threshold: if provided, apply binary thresholding to gating
        """
        # Use the fixed mask directly
        learnable_mask = self.mask

        # Apply the learnable mask to connection weights
        W_masked = self.W * learnable_mask  # Shape: (num_features, num_features)

        # Compute incoming messages: (batch_size, num_features) @ (num_features, num_features) = (batch_size, num_features)
        incoming = torch.matmul(x, W_masked)  # Sum over j for each i: sum_j (x_j * W_ij)

        # Element-wise multiplication with own feature value
        node_input = incoming * x  # Shape: (batch_size, num_features)

        node_input = self.batch_norm(node_input)

        # Add bias
        node_input += self.bias  # Broadcasting over batch

        # Apply Leaky ReLU activation
        node_output = F.leaky_relu(node_input)  # Shape: (batch_size, num_features)

        # Aggregate node outputs to final output
        output = torch.matmul(node_output, self.output_weights)  # Shape: (batch_size,)

        return output  # Raw scores (logits)

# Import ReliefF for feature selection
from skrebate import ReliefF

def global_feature_ranking(X, y):
    """
    Perform global feature ranking using ReliefF.
    """
    # Initialize ReliefF without random_state
    relief = ReliefF(n_features_to_select='all')

    # Convert X and y to NumPy arrays to avoid KeyError
    X_array = X.values
    y_array = y.values
    
    # Fit ReliefF on the entire dataset
    relief.fit(X_array, y_array)
    
    # Get feature scores
    feature_scores = relief.feature_importances_
    
    # Create a DataFrame for easy handling
    feature_score_df = pd.DataFrame({
        'feature': X.columns,
        'score': feature_scores
    })
    
    # Sort features by score in descending order
    feature_score_df = feature_score_df.sort_values(by='score', ascending=False)
    
    return feature_score_df

# ----------------------------
# Cross-Validation with Custom Classifier
# ----------------------------

# Best hyperparameters from your Optuna optimization
best_hyperparams = {
    'n_epochs': 2059,
    'lr': 1.185885489910591e-05,
    'weight_decay': 9.916744984817729e-07,
    'batch_size': 64,
    'n_genotype': 125,
    'n_history': 3,
    'n_phenotype': 44,
    'n_behaviour': 20
}

# ----------------------------
# Global Feature Ranking (Before Cross-Validation)
# ----------------------------
print("Performing global feature ranking on entire dataset...")
global_feature_score_df = global_feature_ranking(X, y)

# Select features based on best hyperparameters (once for all folds)
selected_features = []
for group, n_select in zip(['Genotype', 'History', 'Phenotype', 'Behaviour'],
                          [best_hyperparams['n_genotype'], 
                           best_hyperparams['n_history'],
                           best_hyperparams['n_phenotype'],
                           best_hyperparams['n_behaviour']]):
    group_features = feature_groups[group]
    group_feature_scores = global_feature_score_df[global_feature_score_df['feature'].isin(group_features)]
    top_features = group_feature_scores.head(n_select)['feature'].tolist()
    selected_features.extend(top_features)

# Remove duplicates while preserving order
selected_features = list(dict.fromkeys(selected_features))
print(f"\nSelected {len(selected_features)} features globally:")
print(selected_features)

# Get adjacency matrix for selected features (once for all folds)
selected_feature_indices = [feature_to_idx[feat] for feat in selected_features]
selected_adjacency = adjacency[np.ix_(selected_feature_indices, selected_feature_indices)]
selected_adjacency_mask = torch.tensor(selected_adjacency, dtype=torch.float32)

# ----------------------------
# Cross-Validation with Fixed Feature Set
# ----------------------------
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Storage for metrics and predictions
all_actuals = []
all_probs = []
fold_metrics = {
    'auc': [],
    'auprc': [],
    'f1': [],
    'accuracy': [],
    'precision': [],
    'sensitivity': [],
    'specificity': []
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nTraining Fold {fold+1}/10")
    
    # Split data using pre-selected features
    X_train_fold = X.iloc[train_idx][selected_features].values.astype(np.float32)
    X_test_fold = X.iloc[test_idx][selected_features].values.astype(np.float32)
    y_train_fold = y.iloc[train_idx].values.copy()  # Convert to NumPy array
    y_test_fold = y.iloc[test_idx].values.copy()    # Convert to NumPy array
    
    # Create DataLoaders
    train_dataset = CustomDataset(X_train_fold, y_train_fold)
    test_dataset = CustomDataset(X_test_fold, y_test_fold)
    
    train_loader = DataLoader(train_dataset, 
                            batch_size=best_hyperparams['batch_size'], 
                            shuffle=True)
    test_loader = DataLoader(test_dataset, 
                           batch_size=best_hyperparams['batch_size'], 
                           shuffle=False)
    
    # Initialize model with fixed feature set
    model = CustomNetwork(
        num_features=len(selected_features),
        adjacency_mask=selected_adjacency_mask
    ).to(device)
    
    # Training (same as before)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=best_hyperparams['lr'],
        weight_decay=best_hyperparams['weight_decay']
    )
    
    model.train()
    for epoch in tqdm(range(best_hyperparams['n_epochs']), desc=f"Fold {fold+1} Epochs"):
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    
    # Evaluation
    model.eval()
    fold_probs = []
    fold_actuals = []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            fold_probs.extend(probs.tolist())
            fold_actuals.extend(batch_y.cpu().numpy().flatten().tolist())
    
    # Store results
    all_actuals.extend(fold_actuals)
    all_probs.extend(fold_probs)
    
    # Calculate fold metrics
    fold_auc = roc_auc_score(fold_actuals, fold_probs)
    fold_auprc = average_precision_score(fold_actuals, fold_probs)
    fold_metrics['auc'].append(fold_auc)
    fold_metrics['auprc'].append(fold_auprc)


# ----------------------------
# Find Optimal Threshold and Calculate Final Metrics
# ----------------------------
precision, recall, thresholds = precision_recall_curve(all_actuals, all_probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

print("\nRecalculating metrics with optimal threshold...")
for fold in range(10):
    start = len(all_actuals)//10 * fold
    end = len(all_actuals)//10 * (fold+1)
    fold_actuals = all_actuals[start:end]
    fold_probs = all_probs[start:end]
    
    y_pred = (np.array(fold_probs) >= best_threshold).astype(int)
    
    # Calculate standard metrics
    fold_metrics['f1'].append(f1_score(fold_actuals, y_pred))
    fold_metrics['accuracy'].append(accuracy_score(fold_actuals, y_pred))
    fold_metrics['precision'].append(precision_score(fold_actuals, y_pred, zero_division=0))
    
    # Calculate sensitivity (recall) and specificity
    tn, fp, fn, tp = confusion_matrix(fold_actuals, y_pred).ravel()
    sensitivity = tp / (tp + fn + 1e-9)
    specificity = tn / (tn + fp + 1e-9)
    fold_metrics['sensitivity'].append(sensitivity)
    fold_metrics['specificity'].append(specificity)

# ----------------------------
# Display Final Results
# ----------------------------
def format_metric(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

print("\nFinal Metrics:")
print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"AUC: {format_metric(np.mean(fold_metrics['auc']), np.std(fold_metrics['auc']))}")
print(f"AUPRC: {format_metric(np.mean(fold_metrics['auprc']), np.std(fold_metrics['auprc']))}")
print(f"F1 Score: {format_metric(np.mean(fold_metrics['f1']), np.std(fold_metrics['f1']))}")
print(f"Accuracy: {format_metric(np.mean(fold_metrics['accuracy']), np.std(fold_metrics['accuracy']))}")
print(f"Precision: {format_metric(np.mean(fold_metrics['precision']), np.std(fold_metrics['precision']))}")
print(f"Sensitivity (Recall): {format_metric(np.mean(fold_metrics['sensitivity']), np.std(fold_metrics['sensitivity']))}")
print(f"Specificity: {format_metric(np.mean(fold_metrics['specificity']), np.std(fold_metrics['specificity']))}")

Performing global feature ranking on entire dataset...

Selected 192 features globally:
['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884', 'rs4362400', 'rs3219008', 'rs3218791', 'rs17576', 'rs2289360', 'rs1134170', 'rs2525504', 'rs4919510', 'rs1937810', 'rs1718119', 'rs10484958', 'rs2285053', 'class12_SNP_risk_score', 'rs1800972', 'rs4328262', 'rs1800469', 'rs2306033', 'rs9340799', 'rs17583842', 'rs4244032', 'rs1643821', '

Fold 1 Epochs: 100%|███████████████████████████████████████████████████████████████| 2059/2059 [08:52<00:00,  3.87it/s]



Training Fold 2/10


Fold 2 Epochs: 100%|███████████████████████████████████████████████████████████████| 2059/2059 [08:53<00:00,  3.86it/s]



Training Fold 3/10


...


Training Fold 9/10


Fold 9 Epochs: 100%|███████████████████████████████████████████████████████████████| 2059/2059 [08:52<00:00,  3.87it/s]



Training Fold 10/10


Fold 10 Epochs: 100%|██████████████████████████████████████████████████████████████| 2059/2059 [08:55<00:00,  3.84it/s]


Recalculating metrics with optimal threshold...

Final Metrics:
Optimal Threshold: 0.1899
AUC: 0.7461 ± 0.0249
AUPRC: 0.2876 ± 0.0519
F1 Score: 0.3282 ± 0.0515
Accuracy: 0.8403 ± 0.0183
Precision: 0.2681 ± 0.0461
Sensitivity (Recall): 0.4253 ± 0.0611
Specificity: 0.8819 ± 0.0171
